# 02 - Data Preprocessing.

## Import Libraries

In [12]:

# Import libraries for data handling
from pathlib import Path
import numpy as np
import pandas as pd


## Load Dataset

In [13]:

# Define the dataset column names
columns = [
    "Pregnancies",
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI",
    "DiabetesPedigreeFunction",
    "Age",
    "Outcome"
]

# Find the project folder
project_dir = Path.cwd()

if not (project_dir / "data" / "diabetes.csv").exists():
    project_dir = project_dir.parent

# Load the original dataset
df = pd.read_csv(
    project_dir / "data" / "diabetes.csv",
    header=None,
    names=columns
)

# Display the dataset dimensions and first five records
print("Dataset shape:", df.shape)
df.head()


Dataset shape: (768, 9)


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


## Identify Invalid Zero Measurements

In [14]:

# List measurements where zero represents missing data
invalid_zero_columns = [
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI"
]

# Count invalid zero measurements before preprocessing
zero_counts = (df[invalid_zero_columns] == 0).sum()

# Display the counts
zero_counts


Glucose            5
BloodPressure     35
SkinThickness    227
Insulin          374
BMI               11
dtype: int64

## Separate Features and Target

In [15]:

# Separate the eight input features
X = df.drop(columns=["Outcome"])

# Separate the diabetes outcome labels
y = df["Outcome"]

# Display the dimensions
print("Features shape:", X.shape)
print("Target shape:", y.shape)



Features shape: (768, 8)
Target shape: (768,)


## Train/Test Split

In [16]:

# Import the function for splitting our dataset
from sklearn.model_selection import train_test_split

# Split the data into 80% training and 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Display the size of each dataset
print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)
print("Training labels:", y_train.shape)
print("Testing labels:", y_test.shape)


Training features: (614, 8)
Testing features: (154, 8)
Training labels: (614,)
Testing labels: (154,)


## Verify Class Distribution

In [17]:

# Compare the class proportions in both datasets
class_distribution = pd.DataFrame({
    "Training (%)": y_train.value_counts(normalize=True).sort_index() * 100,
    "Testing (%)": y_test.value_counts(normalize=True).sort_index() * 100
})

# Add readable class labels
class_distribution.index = ["Non-Diabetic", "Diabetic"]

# Display the percentages
class_distribution.round(2)


,Training (%),Testing (%)
Non-Diabetic,65.15,64.94
Diabetic,34.85,35.06


## Replace Invalid Zeros

In [18]:

# Create copies to keep the original datasets unchanged
X_train_clean = X_train.copy()
X_test_clean = X_test.copy()

# Replace invalid zero measurements with missing values
X_train_clean[invalid_zero_columns] = X_train_clean[invalid_zero_columns].replace(0, np.nan)
X_test_clean[invalid_zero_columns] = X_test_clean[invalid_zero_columns].replace(0, np.nan)

# Check missing values in the training data
X_train_clean.isnull().sum()


Pregnancies                   0
Glucose                       4
BloodPressure                23
SkinThickness               175
Insulin                     290
BMI                           9
DiabetesPedigreeFunction      0
Age                           0
dtype: int64

## Median Imputation

In [19]:

# Import the missing value imputer
from sklearn.impute import SimpleImputer

# Create a median-based imputer
imputer = SimpleImputer(strategy="median")

# Learn median values from the training data only
X_train_imputed = pd.DataFrame(
    imputer.fit_transform(X_train_clean),
    columns=X_train.columns,
    index=X_train.index
)

# Apply the same training medians to the testing data
X_test_imputed = pd.DataFrame(
    imputer.transform(X_test_clean),
    columns=X_test.columns,
    index=X_test.index
)

# Verify that no missing values remain
print("Training missing values:", X_train_imputed.isnull().sum().sum())
print("Testing missing values:", X_test_imputed.isnull().sum().sum())


Training missing values: 0
Testing missing values: 0


## Feature Scaling

In [20]:

# Import the standard scaler
from sklearn.preprocessing import StandardScaler

# Create the feature scaler
scaler = StandardScaler()

# Learn scaling parameters from the training data
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train_imputed),
    columns=X_train.columns,
    index=X_train.index
)

# Apply the same scaling to the testing data
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test_imputed),
    columns=X_test.columns,
    index=X_test.index
)

# Preview the scaled training features
X_train_scaled.head()


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
353,-0.851355,-1.056427,-0.826740,-1.918187,-1.203361,-0.769694,0.310794,-0.792169
711,0.356576,0.144399,0.477772,-0.229874,-1.470195,-0.417714,-0.116439,0.561034
373,-0.549372,-0.556083,-1.152868,1.233330,-0.555335,0.359576,-0.764862,-0.707594
46,-0.851355,0.811525,-1.315932,-0.004766,-0.161437,-0.403048,0.262314,-0.369293
682,-1.153338,-0.889646,-0.663676,1.120776,-0.415565,1.782163,-0.337630,-0.961320


## Verify Feature Scaling

In [21]:

# Calculate the mean and standard deviation of scaled features
scaling_summary = pd.DataFrame({
    "Mean": X_train_scaled.mean(),
    "Standard Deviation": X_train_scaled.std(ddof=0)
})

# Display the scaling results
scaling_summary.round(4)


,Mean,Standard Deviation
Pregnancies,-0.0,1.0
Glucose,-0.0,1.0
BloodPressure,0.0,1.0
SkinThickness,-0.0,1.0
Insulin,-0.0,1.0
BMI,0.0,1.0
DiabetesPedigreeFunction,-0.0,1.0
Age,-0.0,1.0


## Final Preprocessing Validation

In [22]:

# Check the processed dataset dimensions
print("Training shape:", X_train_scaled.shape)
print("Testing shape:", X_test_scaled.shape)

# Check for missing values
print("Training missing values:", X_train_scaled.isnull().sum().sum())
print("Testing missing values:", X_test_scaled.isnull().sum().sum())

# Check for infinite values
print("Training infinite values:", np.isinf(X_train_scaled.to_numpy()).sum())
print("Testing infinite values:", np.isinf(X_test_scaled.to_numpy()).sum())


Training shape: (614, 8)
Testing shape: (154, 8)
Training missing values: 0
Testing missing values: 0
Training infinite values: 0
Testing infinite values: 0
